<a href="https://colab.research.google.com/github/khu55/PINN-for-Chem/blob/main/PINN_for_chem_TBBT_decomposition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PINN for Chemistry: TBBT 一级热分解动力学

**目标**：用 Physics-Informed Neural Network (PINN) 拟合 TBBT 的一级分解反应，
核心论点：**在训练数据稀疏（尤其是只覆盖部分时间窗口）的情况下，PINN 比普通神经网络（Plain NN）具有更强的外推（extrapolation）能力**，因为 PINN 在数据之外的区域仍受 ODE 物理约束限制，而 Plain NN 完全依赖数据插值/外推。

## 反应体系

$$\text{TBBT} \xrightarrow{k_1} \text{products}, \quad \frac{dy}{dt} = -k_1 y$$

这是一级反应，解析解已知：

$$y(t) = y_0 \, e^{-k_1 t}$$

**$k_1$ 的来源**：DSC 实验观测到 TBBT 在 154.6°C 分解，半衰期约 300 秒（5分钟），由此反推：

$$k_1 = \frac{\ln 2}{t_{1/2}} = \frac{\ln 2}{300} \approx 2.31\times10^{-3}\ \text{s}^{-1}$$

这个 $k_1$ 有实验依据支撑，不依赖任何额外假设的指前因子 $A$ 或活化能 $E_a$，是整个项目里最干净、最可信的部分——所以本项目只使用这一步反应，确保结论站得住。

## 实验设计

对比不同"训练数据覆盖比例"下，PINN 与 Plain NN 的外推表现：
- 训练数据只覆盖时间窗口的前 X%（X = 10%, 30%, 50%, 70%）
- 在全部时间窗口（0~100%）上评估两个模型，重点看训练区间之外的误差
- 用 MSE/MAE 定量对比，不仅看图


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import pandas as pd

# ── 1. 物理参数（唯一来源：DSC半衰期实验）──────────────
t_half = 300.0                      # 半衰期 300 秒（实验观测值）
k1 = np.log(2) / t_half             # k1 ≈ 2.31e-3 s⁻¹
print(f"k1 = {k1:.4e} s⁻¹  (半衰期 {t_half:.0f} s)")

# ── 2. 时间归一化 ────────────────────────────────────────
# 真实时间窗口选 0~5个半衰期（~1500s），足够看到接近完全分解
T_MAX = 5 * t_half                  # 1500 s
k1_scaled = k1 * T_MAX              # 归一化后的k1（无量纲时间 τ=t/T_MAX 下）
print(f"T_MAX = {T_MAX:.0f} s, k1_scaled = {k1_scaled:.4f}")

# ── 3. 解析解（一级反应有闭式解，用它代替 solve_ivp，更干净也更准）──
def analytic_solution(tau):
    """tau ∈ [0,1] 对应真实时间 t = tau * T_MAX"""
    t_real = tau * T_MAX
    return np.exp(-k1 * t_real)

# 用解析解生成"真值"曲线，替代数值ODE求解（一级反应没必要用solve_ivp，解析解更精确、无数值误差）
tau_full = np.linspace(0, 1, 1000)
y_full = analytic_solution(tau_full)

plt.figure(figsize=(7,4))
plt.plot(tau_full * T_MAX, y_full)
plt.xlabel('Time (s)')
plt.ylabel('Normalized concentration (TBBT)')
plt.title(f'TBBT decomposition, k1={k1:.2e} s⁻¹ (t½={t_half:.0f}s)')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# ── 4. 网络结构（PINN 和 Plain NN 共用同一结构，唯一区别是 loss）──
class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32), nn.Tanh(),
            nn.Linear(32, 32), nn.Tanh(),
            nn.Linear(32, 1),  nn.Sigmoid()   # 输出限制在(0,1)，符合归一化浓度物理意义
        )
    def forward(self, t):
        return self.net(t)


def physics_loss(model, t_phys, k_scaled):
    """
    ODE残差： dy/dtau + k_scaled * y = 0
    （来自 dy/dt = -k1*y，在归一化时间 tau=t/T_MAX 下，链式法则给出 k_scaled = k1*T_MAX）
    """
    t_phys = t_phys.clone().requires_grad_(True)
    y = model(t_phys)
    dy = torch.autograd.grad(y.sum(), t_phys, create_graph=True)[0]
    residual = dy + k_scaled * y
    return (residual ** 2).mean()


def train_plain(t_train, y_train, epochs=3000, lr=1e-3, seed=42):
    torch.manual_seed(seed)
    model = SimpleNN()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        opt.zero_grad()
        loss = nn.MSELoss()(model(t_train), y_train)
        loss.backward()
        opt.step()
    return model


def train_pinn(t_train, y_train, t_phys, k_scaled, epochs=3000, lr=1e-3,
                phys_weight=0.1, seed=42):
    torch.manual_seed(seed)
    model = SimpleNN()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        opt.zero_grad()
        l_data = nn.MSELoss()(model(t_train), y_train)
        l_phys = physics_loss(model, t_phys, k_scaled)
        loss = l_data + phys_weight * l_phys
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    return model


In [ ]:
# ── 5. 核心实验：不同训练数据覆盖比例下的外推对比 ──────────
np.random.seed(0)

coverage_ratios = [0.1, 0.3, 0.5, 0.7]   # 训练数据只覆盖时间窗口的前 X%
n_points_per_ratio = 50                   # 每组实验里训练点的数量（覆盖区间内均匀采样）

# 全时间范围的评估网格（用于算误差和画图）
tau_eval = np.linspace(0, 1, 500)
y_eval_true = analytic_solution(tau_eval)
t_eval_tensor = torch.tensor(tau_eval, dtype=torch.float32).unsqueeze(1)

# 物理约束采样点：固定用全范围（这是PINN能外推的关键——物理约束不受数据覆盖范围限制）
t_phys = torch.linspace(0, 1, 300).unsqueeze(1)

results = []   # 存每组实验的预测结果，后面画图和算指标用

for ratio in coverage_ratios:
    # 训练数据只在 [0, ratio] 区间内采样
    tau_train = np.linspace(0, ratio, n_points_per_ratio)
    y_train_np = analytic_solution(tau_train)
    t_train = torch.tensor(tau_train, dtype=torch.float32).unsqueeze(1)
    y_train = torch.tensor(y_train_np, dtype=torch.float32).unsqueeze(1)

    model_plain = train_plain(t_train, y_train)
    model_pinn  = train_pinn(t_train, y_train, t_phys, k1_scaled)

    model_plain.eval(); model_pinn.eval()
    with torch.no_grad():
        y_pred_plain = model_plain(t_eval_tensor).numpy().flatten()
        y_pred_pinn  = model_pinn(t_eval_tensor).numpy().flatten()

    results.append({
        'ratio': ratio,
        'y_pred_plain': y_pred_plain,
        'y_pred_pinn': y_pred_pinn,
    })

    print(f"训练覆盖比例 {ratio:.0%} 完成 (训练点数={n_points_per_ratio}, "
          f"覆盖时间 0~{ratio*T_MAX:.0f}s / 全程{T_MAX:.0f}s)")


In [ ]:
# ── 6. 可视化：每个覆盖比例下，PINN vs Plain NN vs 真值 ────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for ax, res in zip(axes, results):
    ratio = res['ratio']
    t_plot = tau_eval * T_MAX
    cutoff_time = ratio * T_MAX

    ax.plot(t_plot, y_eval_true, 'k--', linewidth=2, label='真值 (解析解)')
    ax.plot(t_plot, res['y_pred_plain'], 'r-', label='Plain NN')
    ax.plot(t_plot, res['y_pred_pinn'], 'b-', label='PINN')
    ax.axvline(x=cutoff_time, color='gray', linestyle=':', label='训练数据截止点')
    ax.set_title(f'训练数据覆盖前 {ratio:.0%} 区间')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Normalized concentration')
    ax.legend(fontsize=8)
    ax.grid(True)

plt.suptitle('PINN vs Plain NN：不同训练数据覆盖比例下的外推能力对比', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# ── 7. 定量对比：只看外推区间（训练数据覆盖范围之外）的误差 ────────
def compute_extrap_error(y_true, y_pred, tau_eval, ratio):
    """只计算 tau > ratio（也就是训练数据没覆盖到的区间）的误差"""
    mask = tau_eval > ratio
    if mask.sum() == 0:
        return np.nan, np.nan
    mse = np.mean((y_true[mask] - y_pred[mask]) ** 2)
    mae = np.mean(np.abs(y_true[mask] - y_pred[mask]))
    return mse, mae

rows = []
for res in results:
    ratio = res['ratio']
    mse_plain, mae_plain = compute_extrap_error(y_eval_true, res['y_pred_plain'], tau_eval, ratio)
    mse_pinn,  mae_pinn  = compute_extrap_error(y_eval_true, res['y_pred_pinn'],  tau_eval, ratio)
    rows.append({
        '训练覆盖比例': f"{ratio:.0%}",
        'Plain NN 外推MSE': mse_plain,
        'PINN 外推MSE': mse_pinn,
        'MSE改善倍数': mse_plain / mse_pinn if mse_pinn > 0 else np.nan,
        'Plain NN 外推MAE': mae_plain,
        'PINN 外推MAE': mae_pinn,
    })

df_results = pd.DataFrame(rows)
pd.set_option('display.float_format', lambda x: f'{x:.6f}' if abs(x) < 1 else f'{x:.2f}')
print(df_results.to_string(index=False))


In [ ]:
# ── 8. 改善倍数趋势图：训练数据越少，PINN 的优势是否越明显？──────────
plt.figure(figsize=(7, 4.5))
ratios_pct = [r['ratio'] for r in results]
improvement = df_results['MSE改善倍数'].values

plt.plot(ratios_pct, improvement, 'o-', color='darkgreen', linewidth=2, markersize=8)
plt.xlabel('训练数据覆盖比例')
plt.ylabel('外推MSE改善倍数 (Plain NN / PINN)')
plt.title('PINN 相对 Plain NN 的外推优势，随训练数据覆盖比例变化')
plt.axhline(y=1, color='gray', linestyle='--', label='无差异基准线 (=1)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

print("\n结论解读：")
print("- 若改善倍数 > 1，说明在训练数据未覆盖区域，PINN预测误差小于Plain NN")
print("- 若改善倍数随覆盖比例下降而增大，说明训练数据越稀疏，PINN的物理约束优势越明显")
print("  （这是符合预期的：数据越少，Plain NN越容易在外推区间'瞎猜'，而PINN始终受ODE约束）")
